# Pokemon Image Classification – Transfer Learning with ViT

This notebook trains a Vision Transformer (ViT) on a custom Pokemon dataset using transfer learning,
then pushes the model to Hugging Face Hub.

**Dataset classes:** charizard, charmander, charmeleon, ditto, eevee, ekans

## 1. Install dependencies

In [ ]:
# Run once if packages are not installed
# !pip install transformers datasets evaluate torch torchvision huggingface_hub

## 2. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

from huggingface_hub import notebook_login
from datasets import DatasetDict, load_dataset
from transformers import AutoImageProcessor, ViTForImageClassification, Trainer, TrainingArguments
import evaluate

## 3. Login to Hugging Face Hub

> **Action required:** Run this cell and enter your HF token when prompted.
> Get your token at https://huggingface.co/settings/tokens (write access needed).

In [ ]:
notebook_login()

## 4. Load the Pokemon dataset

The dataset is stored locally under `../transferlearning_with_custom_data/data/pokemon/`.
It has a `train/` and `test/` split, each with one subfolder per class.

In [ ]:
dataset = load_dataset(
    "imagefolder",
    data_dir="../transferlearning_with_custom_data/data/pokemon"
)
dataset

In [ ]:
label_names = dataset['train'].features['label'].int2str
labels = dataset['train'].unique('label')
print(f"{len(labels)} classes:", [label_names(l) for l in labels])

### Visualise sample images

In [ ]:
def show_samples(ds, rows=3, cols=5):
    samples = ds.shuffle().select(range(rows * cols))
    fig = plt.figure(figsize=(cols * 3, rows * 3))
    for i in range(rows * cols):
        img   = samples[i]['image']
        label = label_names(samples[i]['label'])
        ax = fig.add_subplot(rows, cols, i + 1)
        plt.imshow(img)
        plt.title(label)
        plt.axis('off')
    plt.tight_layout()
    plt.show()

show_samples(dataset['train'])

## 5. Preprocessing

We split the existing `train` split further into train / validation.  
The original `test` split is kept as our held-out test set.

In [ ]:
split = dataset['train'].train_test_split(test_size=0.15, seed=42)

our_dataset = DatasetDict({
    'train':      split['train'],
    'validation': split['test'],
    'test':       dataset['test'],
})
our_dataset

In [ ]:
label2id = {label_names(c): c for c in labels}
id2label = {c: label_names(c) for c in labels}
print(id2label)

In [ ]:
MODEL_CKPT = 'google/vit-base-patch16-224'
processor  = AutoImageProcessor.from_pretrained(MODEL_CKPT)

def transforms(batch):
    batch['image'] = [img.convert('RGB') for img in batch['image']]
    inputs = processor(batch['image'], return_tensors='pt')
    inputs['labels'] = [label2id[label_names(y)] for y in batch['label']]
    return inputs

processed_dataset = our_dataset.with_transform(transforms)
processed_dataset

In [ ]:
def collate_fn(batch):
    return {
        'pixel_values': torch.stack([x['pixel_values'] for x in batch]),
        'labels':       torch.tensor([x['labels'] for x in batch]),
    }

## 6. Metrics

In [ ]:
accuracy = evaluate.load('accuracy')

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

## 7. Load & fine-tune the model

We freeze all layers except the final classifier head to perform **transfer learning**.

In [ ]:
model = ViTForImageClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

# Freeze everything except the classifier head
for name, p in model.named_parameters():
    if not name.startswith('classifier'):
        p.requires_grad = False

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total:,} | Trainable: {trainable:,}")

In [ ]:
# ⚠️  Replace 'Danydarizzler' with your actual Hugging Face username!
HF_USERNAME  = 'Danydarizzler'
MODEL_REPO   = f'{HF_USERNAME}/pokemon-vit'

training_args = TrainingArguments(
    output_dir='./pokemon-vit',
    per_device_train_batch_size=16,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    logging_steps=20,
    num_train_epochs=10,
    learning_rate=3e-4,
    save_total_limit=2,
    remove_unused_columns=False,
    push_to_hub=True,
    hub_model_id=MODEL_REPO,
    load_best_model_at_end=True,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
    train_dataset=processed_dataset['train'],
    eval_dataset=processed_dataset['validation'],
    tokenizer=processor,
)

trainer.train()

## 8. Evaluate on test set

In [ ]:
results = trainer.evaluate(processed_dataset['test'])
print(results)

## 9. Push model to Hugging Face Hub

In [ ]:
kwargs = {
    'finetuned_from': MODEL_CKPT,
    'dataset': 'pokemon (custom, 6 classes)',
    'tasks': 'image-classification',
    'tags': ['image-classification', 'pokemon'],
}
trainer.save_model()
trainer.push_to_hub('pokemon_vit_exercise', **kwargs)
print(f"Model pushed to: https://huggingface.co/{MODEL_REPO}")

## 10. Quick visual check of predictions

In [ ]:
def show_predictions(rows=3, cols=3):
    samples = our_dataset['test'].shuffle().select(np.arange(rows * cols))
    processed_samples = samples.with_transform(transforms)
    preds = trainer.predict(processed_samples).predictions.argmax(axis=1)

    fig = plt.figure(figsize=(cols * 4, rows * 4))
    for i in range(rows * cols):
        img  = samples[i]['image']
        true = label_names(samples[i]['label'])
        pred = id2label[preds[i]]
        color = 'green' if true == pred else 'red'
        ax = fig.add_subplot(rows, cols, i + 1)
        plt.imshow(img)
        plt.title(f"True: {true}\nPred: {pred}", color=color)
        plt.axis('off')
    plt.tight_layout()
    plt.show()

show_predictions()